# Exp 5 – Erklärbarkeit: KernelSHAP & ShapPFN (airbnb)
- Feature-Attributionen für einen TabPFN-Klassifikator; m = 8 Features (Top-8 nach KernelSHAP), Kontext n = 200 (1:4)
- KernelSHAP = langsamer Gold-Standard; ShapPFN = SHAP als Nebenprodukt des Forward-Pass
- 30 Erklär-Zeilen (seed 42); MLflow → airbnb_paris_experiment_5; Fidelity = Pearson/Spearman/Cosine vs. KernelSHAP

In [9]:
import time
import sys
import numpy as np
import pandas as pd
import mlflow
import shap
import torch  # noqa: F401  (vor weiteren Modell-Imports laden)
from scipy.stats import spearmanr, pearsonr
from huggingface_hub import hf_hub_download
from tabpfn import TabPFNClassifier
sys.path.insert(0, "../../ShapPFN/src")  # geklontes Repo (package-dir = src)
from models import ShapPFNClassifier, ShapPFNModel

## Daten & Subset
- `cleaned`-Features (numerisch); Top-8 nach KernelSHAP-Wichtigkeit aus dem Full-Feature-Lauf
- Deterministisches Kontext-/Erklär-Set (n = 200 / 30, 1:4, seed 42)

In [10]:
LABEL = "is_top_rating"
TEXT_COLS = ['name', 'description', 'neighborhood_overview', 'host_about']
FEAT8 = ["calculated_host_listings_count", "estimated_occupancy_l365d", "minimum_nights", "instant_bookable", "host_tenure_days", "host_is_superhost", "calculated_host_listings_count_entire_homes", "availability_365"]  # Top-8 nach KernelSHAP-Wichtigkeit (Full-Feature-Lauf)

df = pd.read_csv("../../data/preprocessed/cleaned_text_airbnb_paris.csv", keep_default_na=False).set_index("row_id")
y = df[LABEL].astype(int)
Xnum = df[FEAT8]                # m = 8 Features (gemeinsame Obergrenze beider PFN-Explainer)
num_cols = FEAT8
outlier_label = y.value_counts().idxmin()

rng = np.random.RandomState(42)
out_idx = y.index[y == outlier_label].to_numpy()
in_idx = y.index[y != outlier_label].to_numpy()
rng.shuffle(out_idx)
rng.shuffle(in_idx)
ctx_id = np.concatenate([out_idx[:40], in_idx[:160]])           # n = 200 Kontext, 1:4
explain_id = np.concatenate([out_idx[40:50], in_idx[160:180]])  # 10 + 20 = 30 Erklär-Zeilen
outlier_col = sorted(np.unique(y.loc[ctx_id]).tolist()).index(outlier_label)
print("Kontext:", len(ctx_id), "| erklärt:", len(explain_id), "| Features:", Xnum.shape[1])

Kontext: 200 | erklärt: 30 | Features: 8


## KernelSHAP (Baseline)
- TabPFN-Klassifikator fitten, `KernelExplainer` über `P(Outlier)`
- Referenzmatrix + Modellvorhersagen speichern (ExplainerPFN erklärt Vorhersagen, nicht Labels)

In [11]:
tab = TabPFNClassifier()
tab.fit(Xnum.loc[ctx_id].values, y.loc[ctx_id].values)

def predict_outlier(arr):
    return tab.predict_proba(arr)[:, outlier_col]

background = Xnum.loc[ctx_id].sample(10, random_state=42)
explain = Xnum.loc[explain_id]

t0 = time.time()
sv = np.array(shap.KernelExplainer(predict_outlier, background).shap_values(explain, nsamples=100))
t_kernel = time.time() - t0
sv = sv.reshape(len(explain_id), len(num_cols))   # (Zeilen, Features)

imp = pd.Series(np.abs(sv).mean(axis=0), index=num_cols)
display(imp.sort_values(ascending=False).round(5).to_frame("mean_abs_shap"))
print(f"KernelSHAP – t={t_kernel:.1f}s")

# Referenzmatrix (Attributionen) für die Fidelity im ExplainerPFN-Notebook
pd.DataFrame(sv, index=explain_id, columns=num_cols).rename_axis("row_id").to_csv("ref_kernelshap.csv")
# Modellvorhersagen P(Outlier) als Ziel für ExplainerPFN (erklärt Vorhersagen, nicht Ground-Truth)
ids = np.concatenate([ctx_id, explain_id])
pd.Series(tab.predict_proba(Xnum.loc[ids].values)[:, outlier_col], index=ids,
          name="y_pred").rename_axis("row_id").to_csv("tabpfn_preds.csv")

mlflow.set_tracking_uri("file:../../mlruns")
mlflow.set_experiment("airbnb_paris_experiment_5")
with mlflow.start_run(run_name="kernelshap"):
    for c in num_cols:
        mlflow.log_metric(f"num_{c}", float(imp[c]))
    mlflow.log_metric("runtime_s", round(t_kernel, 2))

  0%|          | 0/30 [00:00<?, ?it/s]

,mean_abs_shap
availability_365,0.12194
calculated_host_listings_count,0.06530
host_is_superhost,0.05434
estimated_occupancy_l365d,0.05204
calculated_host_listings_count_entire_homes,0.05143
instant_bookable,0.03591
host_tenure_days,0.03105
minimum_nights,0.02623


KernelSHAP – t=34.9s


## ShapPFN
- Pretrained Checkpoint `shappfn.pth` (HF `Kunumi/ShapPFN`); `explain()` liefert eine `shap.Explanation`
- `.values` hat Form (n, Features, Klassen) → auf die Outlier-Klasse slicen
- Fidelity vs. KernelSHAP: Pearson + Spearman (signiert) + Cosine (|attr|) je Zeile

In [12]:
ckpt = hf_hub_download(repo_id="Kunumi/ShapPFN", filename="shappfn.pth")
net = ShapPFNModel(embedding_size=96, num_attention_heads=4, mlp_hidden_size=192, num_layers=3, num_outputs=2)
net.load_state_dict(torch.load(ckpt, map_location="cpu"))
net.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

clf = ShapPFNClassifier(net, device=device)
clf.fit(Xnum.loc[ctx_id].values.astype(float), y.loc[ctx_id].values)

t0 = time.time()
exp = clf.explain(Xnum.loc[explain_id].values.astype(float))   # shap.Explanation, .values (n, d, C)
t_shappfn = time.time() - t0
shap_pfn = np.asarray(exp.values)[:, :, outlier_col]            # (Zeilen, Features)

imp = pd.Series(np.abs(shap_pfn).mean(axis=0), index=num_cols)
display(imp.sort_values(ascending=False).round(5).to_frame("mean_abs_shap"))

pear = np.mean([pearsonr(sv[i], shap_pfn[i]).statistic for i in range(len(explain_id))])
spear = np.mean([spearmanr(sv[i], shap_pfn[i]).statistic for i in range(len(explain_id))])
cosine = np.mean([np.abs(sv[i]) @ np.abs(shap_pfn[i]) /
                  (np.linalg.norm(sv[i]) * np.linalg.norm(shap_pfn[i]) + 1e-12)
                  for i in range(len(explain_id))])
print(f"ShapPFN – t={t_shappfn:.1f}s | Speedup={t_kernel / t_shappfn:.1f}x")
print(f"Fidelity vs KernelSHAP: pearson={pear:.4f}  spearman={spear:.4f}  cosine={cosine:.4f}")

with mlflow.start_run(run_name="shappfn"):
    for c in num_cols:
        mlflow.log_metric(f"num_{c}", float(imp[c]))
    mlflow.log_metric("runtime_s", round(t_shappfn, 2))
    mlflow.log_metric("pearson_vs_kernelshap", float(pear))
    mlflow.log_metric("spearman_vs_kernelshap", float(spear))
    mlflow.log_metric("cosine_vs_kernelshap", float(cosine))

,mean_abs_shap
availability_365,0.31420
host_tenure_days,0.17374
instant_bookable,0.16269
calculated_host_listings_count,0.13170
calculated_host_listings_count_entire_homes,0.12359
estimated_occupancy_l365d,0.09046
host_is_superhost,0.05609
minimum_nights,0.01166


ShapPFN – t=0.0s | Speedup=6492.7x
Fidelity vs KernelSHAP: pearson=0.8264  spearman=0.7754  cosine=0.8835
